#Install Library

In [1]:
!pip install sastrawi swifter

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 8.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 15.5 MB/s eta 0:00:00
  Created wheel for swifter: filename=swifter-1.4.0-py3-none-any.whl size=16505 sha256=62a1e3c72959d3d393b3a705a6771b5979551ab3a4f2b809bd012ea424677236
  Stored in directory: /root/.cache/pip/wheels/d9/31/ff/ff51141a088571a9f672449e5aad5ea8bb35ca5d95ba135f30
Successfully built swifter


#Import Library

In [2]:
import pandas as pd
import numpy as np
import re
import string
import nltk
import swifter

from nltk.corpus import stopwords

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

#Load Dataset

In [3]:
df = pd.read_csv('https://raw.githubusercontent.com/raihanmeintaro/Dataset/refs/heads/main/NLP/df_combined.csv')

print(df.head())

   Unnamed: 0                                               text  \
0           0                        pagi2 udah di buat emosi :)   
1           1  kok stabilitas negara, memange 10 thn negara t...   
2           2                       dah lah emosi mulu liat emyu   
3           3  aib? bodoh benar! sebelum kata aib itu muncul,...   
4           4                            dih lu yg nyebelin bego   

  emotion_label  stress_label  
0         angry          0.90  
1         angry          0.93  
2         angry          0.95  
3         angry          0.88  
4         angry          1.00  


##Shuffling Dataset

In [4]:
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

##Missing Values Handling

In [5]:
df = df.dropna(subset=['text', 'emotion_label', 'stress_label'])
df = df.reset_index(drop=True)

print(df.isnull().sum())

Unnamed: 0       0
text             0
emotion_label    0
stress_label     0
dtype: int64


#Load Dictionary

##Load Kamus Alay from GitHub

In [6]:
alay_df = pd.read_csv('https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/colloquial-indonesian-lexicon.csv')

# Buat Dictionary
alay_dict = dict(zip(alay_df['slang'], alay_df['formal']))

print("Kamus alay:", len(alay_dict))

Kamus alay: 4331


##Load Additional Kamus Alay

In [7]:
new_alay_df = pd.read_csv('https://raw.githubusercontent.com/okkyibrohim/id-multi-label-hate-speech-and-abusive-language-detection/master/new_kamusalay.csv', encoding='latin-1', names=['slang', 'formal'])

# Buat Dictionary
new_alay_dict = dict(zip(new_alay_df['slang'], new_alay_df['formal']))

print("Typo dictionary:", len(new_alay_dict))

Typo dictionary: 15167


##Merge Kamus Alay Dictionary

In [8]:
combined_dict = {}

combined_dict.update(alay_dict)

combined_dict.update(new_alay_dict)

print("Total combined dictionary:", len(combined_dict))

Total combined dictionary: 15625


##Mapping Emoji Dictionary

In [9]:
emoji_dict = {

    "😭": " sedih ",
    "😢": " sedih ",
    "😔": " sedih ",

    "😡": " marah ",
    "😠": " marah ",

    "😊": " senang ",
    "😁": " senang ",
    "😄": " senang ",

    "😨": " takut ",
    "😰": " cemas ",
    "😥": " cemas "
}

#Pre-Processing

##Casefolding (lower text)

In [10]:
def case_folding(text):

    return text.lower()

##Cleaning Text

In [11]:
def clean_text(text):

    # remove placeholder
    text = re.sub(r'\[.*?\]', ' ', text)

    # remove url
    text = re.sub(r"http\S+", " ", text)

    text = re.sub(r"www\S+", " ", text)

    # remove html
    text = re.sub(r"<.*?>", " ", text)

    # remove mention
    text = re.sub(r"@\w+", " ", text)

    # remove hashtag
    text = re.sub(r"#\w+", " ", text)

    # remove number
    text = re.sub(r"\d+", " ", text)

    # normalize laugh
    text = re.sub(r'(wkwk+|haha+|hehe+)',' lucu ', text)

    # normalize repeated char
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    # remove punctuation
    text = re.sub(
        r'[%s]' % re.escape(string.punctuation), ' ', text)

    # remove non alphabet
    text = re.sub(r'[^a-zA-Z\s]',' ', text)

    # remove extra whitespace
    text = re.sub(r"\s+", " ", text)

    text = text.strip()

    return text

##Additonal Dictionary

In [12]:
custom_normalization = {

    "gk": "tidak",
    "ga": "tidak",
    "nggak": "tidak",
    "tdk": "tidak",

    "bgt": "banget",
    "bgtt": "banget",

    "capek": "lelah",
    "cape": "lelah",

    "ngeri": "takut",

    "ovt": "cemas",
    "overthinking": "cemas",

    "stress": "cemas",

    "gw": "saya",
    "gue": "saya",

    "lu": "kamu",
    "loe": "kamu",

    "yg": "yang",
    "dr": "dari",

    "krn": "karena",
    "karna": "karena",

    "trs": "terus",

    "udh": "sudah",
    "udah": "sudah",

    "jg": "juga",

    "org": "orang"
}

##Typo Normalization

In [13]:
def normalize_typo(text):
    words = text.split()
    normalized_words = []

    for word in words:
        normalized_words.append(
            custom_normalization.get(
                word,
                word
            )
        )

    return " ".join(normalized_words)

##Convert Emoji

In [14]:
def convert_emoji(text):
    for emo, meaning in emoji_dict.items():
        text = text.replace(emo, meaning)
    return text

##Repeat Text

In [15]:
def normalize_repeat(text):
    text = re.sub(
        r'(.)\1{2,}',
        r'\1\1',
        text
    )

    return text

##Text Normalization

In [16]:
def normalize_text(text):
    words = text.split()
    normalized_words = []

    for word in words:
        normalized_words.append(
            combined_dict.get(
                word,
                word
            )
        )

    return " ".join(normalized_words)

##Negation Handling

In [17]:
negation_words = [
    'tidak',
    'bukan',
    'jangan',
    'sangat'
]

def handle_negation(text):
    words = text.split()
    result = []

    i = 0
    while i < len(words):
        if (words[i] in negation_words and i+1 < len(words)):
            result.append(words[i] + "_" + words[i+1])
            i += 2
        else:
            result.append(words[i])
            i += 1

    return " ".join(result)

##Remove Single Char

In [18]:
def remove_single_char(text):
    words = text.split()
    filtered = [word for word in words if len(word) > 1]

    return " ".join(filtered)

##Drop Stop Words Except Negation Words

In [19]:
stop_words = set(stopwords.words('indonesian'))

social_stopwords = {
    'nih',
    'sih',
    'dong',
    'loh',
    'kok',
    'deh',
    'lah',
    'nya',
    'aja',
    'yah',

    'wkwk',
    'wkwkwk',

    'hehe',
    'haha'
}

stop_words = stop_words - set(negation_words)
stop_words = stop_words.union(social_stopwords)

def remove_stopwords(text):
    words = text.split()
    filtered_words = [word for word in words if word not in stop_words]

    return " ".join(filtered_words)

##Stemming (Sastrawi)

In [20]:
factory = StemmerFactory()
stemmer = factory.create_stemmer()

def stemming(text):
    return stemmer.stem(text)

##Main Pipeline

In [21]:
def preprocess_text(text):
    text = str(text)
    text = case_folding(text)
    text = normalize_typo(text)
    text = convert_emoji(text)
    text = clean_text(text)
    text = normalize_repeat(text)
    text = normalize_text(text)
    text = handle_negation(text)
    text = remove_stopwords(text)
    text = remove_single_char(text)
    text = stemming(text)

    return text

In [22]:
df['clean_text'] = df['text'].swifter.apply(preprocess_text)

print(df[['text', 'clean_text']].head())

Pandas Apply:   0%|          | 0/31320 [00:00<?, ?it/s]

                                                text  \
0  yang namanya terlanjur sayang itu ribet loh. m...   
1  orang itu bikin emosi banget tapi yaudahlah di...   
2  kesel banget sama orang yang ga tepat waktu pa...   
3  marah banget karena kerjaan berantakan agak ra...   
4  habis beresin kamar dan lanjut kerja banget ta...   

                                          clean_text  
0  nama lanjur sayang ribet cuek tidak bisa marah...  
1                 orang bikin emosi banget ya kampus  
2   kesal banget orang tidak tepat coba santai rumah  
3   marah banget kerja beranta random rumah keluarga  
4                habis beres kamar kerja banget pagi  


In [23]:
df = df[df['clean_text'].str.strip() != '']

df = df.reset_index(drop=True)

#Feature Selection

In [24]:
df = df[['clean_text', 'emotion_label', 'stress_label']]

##Distribusi Label

In [25]:
df.value_counts('emotion_label')

,count
emotion_label,
neutral,6981
happy,6274
angry,6129
sad,5992
anxious,5911


##Convert Dataset

In [26]:
df.to_csv('Preprocessed_Dataset.csv', index=False)